In [1]:
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
import matplotlib.dates as mdates
from datetime import datetime, timedelta
import heapq
import itertools
import time
from abc import ABC, abstractmethod
from collections import deque

# ==========================================
# 1. CORE ENGINE (AdvancedOrderBook)
# ==========================================

class AdvancedOrderBook:
    def __init__(self):
        self.bids = []  
        self.asks = []  
        self.trades = []
        self.order_id_counter = itertools.count() 

    def submit_order(self, side, qty, price=None, order_type='limit'):
        if order_type == 'market':
            limit_price = float('inf') if side == 'buy' else 0
        else:
            limit_price = price

        remaining_qty = self.match(side, qty, limit_price)

        if remaining_qty > 0 and order_type == 'limit':
            entry_id = next(self.order_id_counter)
            if side == 'buy':
                heapq.heappush(self.bids, [-limit_price, entry_id, remaining_qty])
            else:
                heapq.heappush(self.asks, [limit_price, entry_id, remaining_qty])
            
        return remaining_qty

    def match(self, side, qty, limit_price):
        remaining_qty = qty
        while remaining_qty > 0:
            if side == 'buy':
                if not self.asks: break
                best_price = self.asks[0][0]
                if limit_price < best_price: break
                best_order = self.asks[0]
            else:
                if not self.bids: break
                best_price = -self.bids[0][0] 
                if limit_price > best_price: break
                best_order = self.bids[0]

            trade_qty = min(remaining_qty, best_order[2])
            exec_price = best_order[0] if side == 'buy' else -best_order[0]

            self.trades.append({
                'price': exec_price,
                'qty': trade_qty,
                'timestamp': time.time(),
                'side': side,
                'aggressor': 'market' if limit_price == float('inf') else 'limit'
            })

            remaining_qty -= trade_qty
            best_order[2] -= trade_qty

            if best_order[2] == 0:
                if side == 'buy': heapq.heappop(self.asks)
                else: heapq.heappop(self.bids)
                    
        return remaining_qty

# ==========================================
# 2. ANALYTICS ENGINE
# ==========================================

class AnalyticsEngine:
    def __init__(self):
        self.tape = []      
        self.snapshots = [] 

    def log_trade(self, timestamp, price, qty):
        self.tape.append({
            'timestamp': timestamp,
            'price': price,
            'qty': qty
        })

    def log_snapshot(self, timestamp, best_bid, best_ask):
        spread = (best_ask - best_bid) if (best_bid and best_ask) else np.nan
        mid_price = (best_ask + best_bid) / 2 if (best_bid and best_ask) else np.nan
        self.snapshots.append({
            'timestamp': timestamp,
            'spread': spread,
            'mid_price': mid_price
        })

    def get_tape_dataframe(self):
        df = pd.DataFrame(self.tape)
        if not df.empty:
            df['timestamp'] = pd.to_datetime(df['timestamp'])
            df.set_index('timestamp', inplace=True)
        return df

    def get_snapshot_dataframe(self):
        df = pd.DataFrame(self.snapshots)
        if not df.empty:
            df['timestamp'] = pd.to_datetime(df['timestamp'])
            df.set_index('timestamp', inplace=True)
        return df

# ==========================================
# 3. AGENT ZOO (Noise, Momentum, MarketMaker)
# ==========================================

class Agent(ABC):
    def __init__(self, agent_id, cash, inventory):
        self.agent_id = agent_id
        self.cash = cash
        self.inventory = inventory
        
    @abstractmethod
    def get_action(self, market_snapshot, fair_value=None):
        pass

class NoiseTrader(Agent):
    """ Day 7: Liquidity Consumer """
    def __init__(self, agent_id, cash, inventory, arrival_rate=0.5, volatility=0.02):
        super().__init__(agent_id, cash, inventory)
        self.arrival_rate = arrival_rate
        self.volatility = volatility 

    def get_action(self, market_snapshot, fair_value):
        if np.random.random() > self.arrival_rate: return None
        side = 'buy' if np.random.random() > 0.5 else 'sell'
        noise = np.random.normal(0, self.volatility * fair_value)
        order_price = round(fair_value + noise, 2)
        qty = np.random.randint(1, 5)
        return {'agent_id': self.agent_id, 'side': side, 'price': order_price, 'qty': qty, 'type': 'limit'}

class MomentumTrader(Agent):
    """ Day 8: Trend Follower (Instability Source) """
    def __init__(self, agent_id, cash, inventory, lookback=20, threshold=0.5):
        super().__init__(agent_id, cash, inventory)
        self.lookback = lookback
        self.threshold = threshold
        self.price_history = deque(maxlen=lookback)

    def get_action(self, market_snapshot, fair_value=None):
        mid_price = market_snapshot.get('mid_price')
        if mid_price is None: return None
        self.price_history.append(mid_price)
        if len(self.price_history) < self.lookback: return None

        sma = sum(self.price_history) / len(self.price_history)
        qty = 10 
        
        if mid_price > sma + self.threshold:
            return {'agent_id': self.agent_id, 'side': 'buy', 'price': round(mid_price + 0.5, 2), 'qty': qty, 'type': 'limit'}
        elif mid_price < sma - self.threshold:
            return {'agent_id': self.agent_id, 'side': 'sell', 'price': round(mid_price - 0.5, 2), 'qty': qty, 'type': 'limit'}
        return None

class MarketMakerAgent(Agent):
    """ Day 9: Liquidity Provider (Stability Source) """
    def __init__(self, agent_id, cash, inventory, half_spread=0.5, skew_factor=0.05):
        super().__init__(agent_id, cash, inventory)
        self.half_spread = half_spread
        self.skew_factor = skew_factor 

    def get_action(self, market_snapshot, fair_value=None):
        mid_price = market_snapshot.get('mid_price')
        if mid_price is None: return None 

        skew = -1 * self.inventory * self.skew_factor
        bid_price = round(mid_price - self.half_spread + skew, 2)
        ask_price = round(mid_price + self.half_spread + skew, 2)
        
        if bid_price >= ask_price: ask_price = bid_price + 0.01

        return [
            {'agent_id': self.agent_id, 'side': 'buy', 'price': bid_price, 'qty': 5, 'type': 'limit'},
            {'agent_id': self.agent_id, 'side': 'sell', 'price': ask_price, 'qty': 5, 'type': 'limit'}
        ]

# ==========================================
# 4. SCENARIO MANAGER
# ==========================================

# Constants
SEED = 42
NUM_STEPS = 2000
OUTPUT_FILENAME = "week2_final_report.pdf"

# --- SCENARIO DEFINITIONS ---
SCENARIOS = {
    'A': {'noise': 50, 'momentum': 0, 'mm': 0, 'desc': 'Scenario A: Noise Only (High Spread)'},
    'B': {'noise': 50, 'momentum': 0, 'mm': 5, 'desc': 'Scenario B: Noise + Market Makers (Stable)'},
    'C': {'noise': 50, 'momentum': 10, 'mm': 0, 'desc': 'Scenario C: Noise + Momentum (Unstable)'}
}

def generate_fair_value_series(steps, start_price=100.0, drift=0.0, vol=0.0005):
    prices = [start_price]
    for _ in range(steps):
        change = np.random.normal(drift, vol)
        prices.append(prices[-1] * (1 + change))
    return prices

def run_scenario(scenario_name, config):
    print(f"\n--- Running {config['desc']} ---")
    random.seed(SEED)
    np.random.seed(SEED)
    
    engine = AdvancedOrderBook()
    analytics = AnalyticsEngine()
    fair_values = generate_fair_value_series(NUM_STEPS, start_price=100.0)
    
    # Populate Agents based on Config
    agents = []
    agent_id = 0
    
    for _ in range(config['noise']):
        agents.append(NoiseTrader(agent_id, 100000, 1000))
        agent_id += 1
        
    for _ in range(config['mm']):
        agents.append(MarketMakerAgent(agent_id, 500000, 0))
        agent_id += 1
        
    for _ in range(config['momentum']):
        agents.append(MomentumTrader(agent_id, 100000, 0))
        agent_id += 1
        
    current_time = datetime(2025, 1, 1, 9, 30)
    
    for i in range(NUM_STEPS):
        current_time += timedelta(seconds=1)
        current_fair_value = fair_values[i]
        
        # Snapshot
        best_bid = -engine.bids[0][0] if engine.bids else None
        best_ask = engine.asks[0][0] if engine.asks else None
        mid_price = (best_bid + best_ask) / 2 if (best_bid and best_ask) else current_fair_value
        
        analytics.log_snapshot(current_time, best_bid, best_ask)
        snapshot = {'best_bid': best_bid, 'best_ask': best_ask, 'mid_price': mid_price}
        
        # Agents Act
        random.shuffle(agents)
        for agent in agents:
            actions = agent.get_action(snapshot, current_fair_value)
            
            if isinstance(actions, list): # Market Maker returns list
                for order in actions:
                    engine.submit_order(order['side'], order['qty'], order['price'], order['type'])
                    if order['side'] == 'buy': agent.inventory += 1
                    else: agent.inventory -= 1
            elif actions:
                engine.submit_order(actions['side'], actions['qty'], actions['price'], actions['type'])

        # Log Trades
        while engine.trades:
            trade = engine.trades.pop(0)
            analytics.log_trade(current_time, trade['price'], trade['qty'])

    return analytics

def generate_pdf_report(results):
    print(f"Generating Final Report: {OUTPUT_FILENAME}...")
    
    with PdfPages(OUTPUT_FILENAME) as pdf:
        # Title Page
        fig_title = plt.figure(figsize=(8, 6))
        plt.text(0.5, 0.6, "Week 2: Market Engine & Agent Zoo", ha='center', fontsize=20, weight='bold')
        plt.text(0.5, 0.4, "Comparison: Noise, Stability, Instability", ha='center', fontsize=14)
        plt.axis('off')
        pdf.savefig(fig_title)
        plt.close()

        # Iterate through scenarios
        for name, analytics in results.items():
            df_snap = analytics.get_snapshot_dataframe()
            df_tape = analytics.get_tape_dataframe()
            
            fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 8))
            
            # Price Plot
            desc = SCENARIOS[name]['desc']
            ax1.set_title(f"{desc}: Price Action")
            
            if not df_tape.empty:
                # Resample price for cleaner plot
                price_series = df_tape['price'].resample('10s').mean()
                ax1.plot(price_series.index, price_series.values, label='Trade Price', color='blue')
            
            # Overlay Mid Price
            mid_series = df_snap['mid_price'].resample('10s').mean()
            ax1.plot(mid_series.index, mid_series.values, label='Mid Price', color='black', alpha=0.5, linestyle='--')
            ax1.legend()
            ax1.set_ylabel("Price")
            
            # Spread Plot
            spread_series = df_snap['spread'].resample('10s').mean()
            ax2.plot(spread_series.index, spread_series.values, color='red', label='Bid-Ask Spread')
            ax2.set_ylabel("Spread")
            ax2.legend()
            ax2.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
            
            plt.tight_layout()
            pdf.savefig(fig)
            plt.close()
            print(f"Scenario {name} plotted.")
            
        # Comparison Table Page
        fig_table, ax_table = plt.subplots(figsize=(10, 4))
        ax_table.axis('tight')
        ax_table.axis('off')
        
        table_data = [['Scenario', 'Avg Spread', 'Volatility (Std Dev)', 'Total Trades']]
        
        for name, analytics in results.items():
            df_snap = analytics.get_snapshot_dataframe()
            df_tape = analytics.get_tape_dataframe()
            
            avg_spread = df_snap['spread'].mean()
            volatility = df_snap['mid_price'].std() if not df_snap.empty else 0
            trades = len(df_tape)
            
            table_data.append([name, f"{avg_spread:.4f}", f"{volatility:.4f}", trades])
            
        ax_table.table(cellText=table_data, loc='center', cellLoc='center', colWidths=[0.3, 0.2, 0.3, 0.2])
        ax_table.set_title("Scenario Performance Comparison", fontsize=16)
        
        pdf.savefig(fig_table)
        plt.close()

if __name__ == "__main__":
    results = {}
    
    # Run all 3 Scenarios
    for scenario_key in SCENARIOS:
        analytics_result = run_scenario(scenario_key, SCENARIOS[scenario_key])
        results[scenario_key] = analytics_result
    
    generate_pdf_report(results)
    print("Done. Report saved.")


--- Running Scenario A: Noise Only (High Spread) ---

--- Running Scenario B: Noise + Market Makers (Stable) ---

--- Running Scenario C: Noise + Momentum (Unstable) ---
Generating Final Report: week2_final_report.pdf...
Scenario A plotted.
Scenario B plotted.
Scenario C plotted.
Done. Report saved.
